In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, r2_score


In [2]:
df = pd.read_csv("C:\\Users\\HARSH GOSALIA\\Downloads\\retail_sales_bigmart.csv")
df.columns

Index(['Item_Identifier', 'Item_Weight', 'Item_Fat_Content', 'Item_Visibility',
       'Item_Type', 'Item_MRP', 'Outlet_Identifier',
       'Outlet_Establishment_Year', 'Outlet_Size', 'Outlet_Location_Type',
       'Outlet_Type', 'Item_Outlet_Sales'],
      dtype='str')

In [3]:
df.head()


,Item_Identifier,Item_Weight,Item_Fat_Content,Item_Visibility,Item_Type,Item_MRP,Outlet_Identifier,Outlet_Establishment_Year,Outlet_Size,Outlet_Location_Type,Outlet_Type,Item_Outlet_Sales
0,DRC66,5.242170,low fat,0.064002,Dairy,121.71,OUT045,2002,NaN,Tier 2,Supermarket Type1,2419.82
1,DUS55,NaN,LF,0.042572,Snack Foods,203.31,OUT046,2013,Small,Tier 1,Supermarket Type1,2259.98
2,DUJ83,21.058597,Low Fat,0.068903,Snack Foods,237.89,OUT013,1987,High,Tier 3,Supermarket Type1,5870.58
3,DUN02,7.276768,LF,0.038148,Soft Drinks,206.00,OUT013,1987,High,Tier 3,Supermarket Type1,4661.53
4,NCW27,17.113985,reg,0.040903,Breakfast,246.30,OUT018,2009,Medium,Tier 3,Supermarket Type2,3059.90


In [4]:
df.shape

(16000, 12)

In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 16000 entries, 0 to 15999
Data columns (total 12 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   Item_Identifier            16000 non-null  str    
 1   Item_Weight                14414 non-null  float64
 2   Item_Fat_Content           16000 non-null  str    
 3   Item_Visibility            16000 non-null  float64
 4   Item_Type                  16000 non-null  str    
 5   Item_MRP                   16000 non-null  float64
 6   Outlet_Identifier          16000 non-null  str    
 7   Outlet_Establishment_Year  16000 non-null  int64  
 8   Outlet_Size                11861 non-null  str    
 9   Outlet_Location_Type       16000 non-null  str    
 10  Outlet_Type                16000 non-null  str    
 11  Item_Outlet_Sales          16000 non-null  float64
dtypes: float64(4), int64(1), str(7)
memory usage: 1.5 MB


In [6]:
df.isnull().sum()

Item_Identifier                 0
Item_Weight                  1586
Item_Fat_Content                0
Item_Visibility                 0
Item_Type                       0
Item_MRP                        0
Outlet_Identifier               0
Outlet_Establishment_Year       0
Outlet_Size                  4139
Outlet_Location_Type            0
Outlet_Type                     0
Item_Outlet_Sales               0
dtype: int64

Categorical Columns


In [7]:
Categorial_columns = df.select_dtypes(include='str').columns
print("Categorial features:")
print(Categorial_columns)

Categorial features:
Index(['Item_Identifier', 'Item_Fat_Content', 'Item_Type', 'Outlet_Identifier',
       'Outlet_Size', 'Outlet_Location_Type', 'Outlet_Type'],
      dtype='str')


In [8]:
Categorial_columns.isnull().sum()

np.int64(0)

Numeric Colums


In [9]:
numeric_columns = df.select_dtypes(include=np.number).columns
print("Numeric features:")  
numeric_columns

Numeric features:


Index(['Item_Weight', 'Item_Visibility', 'Item_MRP',
       'Outlet_Establishment_Year', 'Item_Outlet_Sales'],
      dtype='str')

In [10]:
numeric_columns.isnull().sum()

np.int64(0)

Fills Null / missing values


In [11]:
df['Item_Weight'] = df.groupby('Item_Type')['Item_Weight']\
                     .transform(lambda x: x.fillna(x.median()))
print(df.isnull().sum())

Item_Identifier                 0
Item_Weight                     0
Item_Fat_Content                0
Item_Visibility                 0
Item_Type                       0
Item_MRP                        0
Outlet_Identifier               0
Outlet_Establishment_Year       0
Outlet_Size                  4139
Outlet_Location_Type            0
Outlet_Type                     0
Item_Outlet_Sales               0
dtype: int64


In [12]:
df['Outlet_Size'] = df.groupby('Outlet_Type')['Outlet_Size'] \
    .transform(lambda x: x.fillna(x.mode()[0] if not x.mode().empty else 'Medium'))
print(df.isnull().sum())

Item_Identifier              0
Item_Weight                  0
Item_Fat_Content             0
Item_Visibility              0
Item_Type                    0
Item_MRP                     0
Outlet_Identifier            0
Outlet_Establishment_Year    0
Outlet_Size                  0
Outlet_Location_Type         0
Outlet_Type                  0
Item_Outlet_Sales            0
dtype: int64


Finding Multiples row with value count


In [13]:
df['Outlet_Size'].value_counts()

Outlet_Size
Small     8897
Medium    4550
High      1347
SMALL      294
medium     270
MED        252
small      246
Large       74
HIGH        70
Name: count, dtype: int64

In [14]:
df['Outlet_Size'] = df['Outlet_Size'].replace({
    'small': 'Small',
    'medium': 'Medium',
    'med': 'Medium',
    'high': 'High',
    'large': 'High',
    'SMALL': 'Small',
    'MED': 'Medium',
    'Large' : 'High',
    'HIGH' : 'High'

})
df['Outlet_Size'].value_counts()

Outlet_Size
Small     9143
Medium    5072
High      1491
SMALL      294
Name: count, dtype: int64

In [15]:
df['Outlet_Size'] = df['Outlet_Size'].replace({
    'small': 'Small',
    'SMALL ' : 'Small',


})

df['Outlet_Size'].value_counts()

Outlet_Size
Small     9437
Medium    5072
High      1491
Name: count, dtype: int64

In [16]:
df['Item_Fat_Content'].value_counts()

Item_Fat_Content
Low Fat     6686
Regular     4816
LF          1457
low fat      952
reg          928
Regular      642
 Low Fat     519
Name: count, dtype: int64

In [17]:
df = df.replace({'low fat' : 'Low Fat', 'LF' : 'Low Fat', 'reg' : 'Regular', ' Low Fat' : 'Low Fat' },inplace=True)
df['Item_Fat_Content'].value_counts()

Item_Fat_Content
Low Fat     9614
Regular     5744
Regular      642
Name: count, dtype: int64

In [18]:
df

,Item_Identifier,Item_Weight,Item_Fat_Content,Item_Visibility,Item_Type,Item_MRP,Outlet_Identifier,Outlet_Establishment_Year,Outlet_Size,Outlet_Location_Type,Outlet_Type,Item_Outlet_Sales
0,DRC66,5.242170,Low Fat,0.064002,Dairy,121.71,OUT045,2002,Small,Tier 2,Supermarket Type1,2419.82
1,DUS55,14.586506,Low Fat,0.042572,Snack Foods,203.31,OUT046,2013,Small,Tier 1,Supermarket Type1,2259.98
2,DUJ83,21.058597,Low Fat,0.068903,Snack Foods,237.89,OUT013,1987,High,Tier 3,Supermarket Type1,5870.58
3,DUN02,7.276768,Low Fat,0.038148,Soft Drinks,206.00,OUT013,1987,High,Tier 3,Supermarket Type1,4661.53
4,NCW27,17.113985,Regular,0.040903,Breakfast,246.30,OUT018,2009,Medium,Tier 3,Supermarket Type2,3059.90
...,...,...,...,...,...,...,...,...,...,...,...,...
15995,DRC12,17.233114,Low Fat,0.010963,Hard Drinks,199.25,OUT013,1987,High,Tier 3,Supermarket Type1,5885.54
15996,NCW46,10.174233,Regular,0.052483,Soft Drinks,289.15,OUT013,1987,High,Tier 3,Supermarket Type1,4847.83
15997,DUN71,10.959149,Low Fat,0.014597,Breakfast,101.59,OUT035,2004,Small,Tier 2,Supermarket Type1,1621.67
15998,NCL16,16.209470,Low Fat,0.053749,Fruits and Vegetables,219.90,OUT049,1999,Medium,Tier 1,Supermarket Type1,3519.60


FEATURE ENGINEERING


In [19]:
df['Price_per_Weight'] = df['Item_MRP'] / df['Item_Weight']

In [20]:
df['Visibility_Mean_Item'] = df.groupby('Item_Type')['Item_Visibility'].transform('mean')
df['Visibility_Ratio'] = df['Item_Visibility'] / df['Visibility_Mean_Item']

In [21]:
df['Outlet_Age'] = 2026 - df['Outlet_Establishment_Year']

In [22]:
df['Item_Category'] = df['Item_Identifier'].str[:2]

In [23]:
df['MRP_Level'] = pd.cut(df['Item_MRP'], bins=4, labels=['Low', 'Medium', 'High', 'Very High'])

In [24]:
df['Outlet_Type_Size'] = df['Outlet_Type'] + "_" + df['Outlet_Size']

In [25]:
df

,Item_Identifier,Item_Weight,Item_Fat_Content,Item_Visibility,Item_Type,Item_MRP,Outlet_Identifier,Outlet_Establishment_Year,Outlet_Size,Outlet_Location_Type,Outlet_Type,Item_Outlet_Sales,Price_per_Weight,Visibility_Mean_Item,Visibility_Ratio,Outlet_Age,Item_Category,MRP_Level,Outlet_Type_Size
0,DRC66,5.242170,Low Fat,0.064002,Dairy,121.71,OUT045,2002,Small,Tier 2,Supermarket Type1,2419.82,23.217487,0.052021,1.230300,24,DR,Low,Supermarket Type1_Small
1,DUS55,14.586506,Low Fat,0.042572,Snack Foods,203.31,OUT046,2013,Small,Tier 1,Supermarket Type1,2259.98,13.938225,0.052874,0.805152,13,DU,Medium,Supermarket Type1_Small
2,DUJ83,21.058597,Low Fat,0.068903,Snack Foods,237.89,OUT013,1987,High,Tier 3,Supermarket Type1,5870.58,11.296574,0.052874,1.303143,39,DU,Medium,Supermarket Type1_High
3,DUN02,7.276768,Low Fat,0.038148,Soft Drinks,206.00,OUT013,1987,High,Tier 3,Supermarket Type1,4661.53,28.309272,0.055857,0.682962,39,DU,Medium,Supermarket Type1_High
4,NCW27,17.113985,Regular,0.040903,Breakfast,246.30,OUT018,2009,Medium,Tier 3,Supermarket Type2,3059.90,14.391739,0.054748,0.747115,17,NC,Medium,Supermarket Type2_Medium
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15995,DRC12,17.233114,Low Fat,0.010963,Hard Drinks,199.25,OUT013,1987,High,Tier 3,Supermarket Type1,5885.54,11.562043,0.053793,0.203799,39,DR,Medium,Supermarket Type1_High
15996,NCW46,10.174233,Regular,0.052483,Soft Drinks,289.15,OUT013,1987,High,Tier 3,Supermarket Type1,4847.83,28.419833,0.055857,0.939601,39,NC,High,Supermarket Type1_High
15997,DUN71,10.959149,Low Fat,0.014597,Breakfast,101.59,OUT035,2004,Small,Tier 2,Supermarket Type1,1621.67,9.269881,0.054748,0.266622,22,DU,Low,Supermarket Type1_Small
15998,NCL16,16.209470,Low Fat,0.053749,Fruits and Vegetables,219.90,OUT049,1999,Medium,Tier 1,Supermarket Type1,3519.60,13.566144,0.052406,1.025622,27,NC,Medium,Supermarket Type1_Medium


In [26]:
df = df.drop([
    'Item_Identifier',
    'Outlet_Identifier',
    'Visibility_Mean_Item',
    'Item_Visibility',
    'Outlet_Establishment_Year',
    'Item_Type'   # optional
], axis=1)

In [27]:
df

,Item_Weight,Item_Fat_Content,Item_MRP,Outlet_Size,Outlet_Location_Type,Outlet_Type,Item_Outlet_Sales,Price_per_Weight,Visibility_Ratio,Outlet_Age,Item_Category,MRP_Level,Outlet_Type_Size
0,5.242170,Low Fat,121.71,Small,Tier 2,Supermarket Type1,2419.82,23.217487,1.230300,24,DR,Low,Supermarket Type1_Small
1,14.586506,Low Fat,203.31,Small,Tier 1,Supermarket Type1,2259.98,13.938225,0.805152,13,DU,Medium,Supermarket Type1_Small
2,21.058597,Low Fat,237.89,High,Tier 3,Supermarket Type1,5870.58,11.296574,1.303143,39,DU,Medium,Supermarket Type1_High
3,7.276768,Low Fat,206.00,High,Tier 3,Supermarket Type1,4661.53,28.309272,0.682962,39,DU,Medium,Supermarket Type1_High
4,17.113985,Regular,246.30,Medium,Tier 3,Supermarket Type2,3059.90,14.391739,0.747115,17,NC,Medium,Supermarket Type2_Medium
...,...,...,...,...,...,...,...,...,...,...,...,...,...
15995,17.233114,Low Fat,199.25,High,Tier 3,Supermarket Type1,5885.54,11.562043,0.203799,39,DR,Medium,Supermarket Type1_High
15996,10.174233,Regular,289.15,High,Tier 3,Supermarket Type1,4847.83,28.419833,0.939601,39,NC,High,Supermarket Type1_High
15997,10.959149,Low Fat,101.59,Small,Tier 2,Supermarket Type1,1621.67,9.269881,0.266622,22,DU,Low,Supermarket Type1_Small
15998,16.209470,Low Fat,219.90,Medium,Tier 1,Supermarket Type1,3519.60,13.566144,1.025622,27,NC,Medium,Supermarket Type1_Medium


In [ ]:

from sklearn.preprocessing import OrdinalEncoder

# Target separate
y = df['Item_Outlet_Sales']
X = df.drop('Item_Outlet_Sales', axis=1)


oe = OrdinalEncoder(categories=[
    ['Small', 'Medium', 'High'],
    ['Low', 'Medium', 'High', 'Very High']
])

X[['Outlet_Size', 'MRP_Level']] = oe.fit_transform(
    X[['Outlet_Size', 'MRP_Level']]
)


X = pd.get_dummies(X, columns=[         # One-Hot Encoding
    'Item_Fat_Content',
    'Outlet_Location_Type',
    'Outlet_Type',
    'Item_Category',
    'Outlet_Type_Size'
], drop_first=True)

print(X.shape)

(16000, 22)


In [44]:
df['Item_Fat_Content'].replace({
    
    'Regular ': 'Regular',
},inplace=True).value_counts()



C:\Users\HARSH GOSALIA\AppData\Local\Temp\ipykernel_5556\129943434.py:1: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  df['Item_Fat_Content'].replace({


Item_Fat_Content
Low Fat    9614
Regular    6386
Name: count, dtype: int64

In [30]:
X[['Outlet_Size', 'MRP_Level']].value_counts()

Outlet_Size  MRP_Level
0.0          1.0          4474
             0.0          3126
1.0          1.0          2410
0.0          2.0          1685
1.0          0.0          1639
             2.0           947
2.0          1.0           704
             0.0           500
             2.0           267
0.0          3.0           152
1.0          3.0            76
2.0          3.0            20
Name: count, dtype: int64

In [31]:
X

,Item_Weight,Item_MRP,Outlet_Size,Price_per_Weight,Visibility_Ratio,Outlet_Age,MRP_Level,Item_Fat_Content_Regular,Item_Fat_Content_Regular,Outlet_Location_Type_Tier 2,...,Outlet_Type_Supermarket Type2,Outlet_Type_Supermarket Type3,Item_Category_DU,Item_Category_FD,Item_Category_NC,Outlet_Type_Size_Supermarket Type1_High,Outlet_Type_Size_Supermarket Type1_Medium,Outlet_Type_Size_Supermarket Type1_Small,Outlet_Type_Size_Supermarket Type2_Medium,Outlet_Type_Size_Supermarket Type3_Medium
0,5.242170,121.71,0.0,23.217487,1.230300,24,0.0,False,False,True,...,False,False,False,False,False,False,False,True,False,False
1,14.586506,203.31,0.0,13.938225,0.805152,13,1.0,False,False,False,...,False,False,True,False,False,False,False,True,False,False
2,21.058597,237.89,2.0,11.296574,1.303143,39,1.0,False,False,False,...,False,False,True,False,False,True,False,False,False,False
3,7.276768,206.00,2.0,28.309272,0.682962,39,1.0,False,False,False,...,False,False,True,False,False,True,False,False,False,False
4,17.113985,246.30,1.0,14.391739,0.747115,17,1.0,True,False,False,...,True,False,False,False,True,False,False,False,True,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15995,17.233114,199.25,2.0,11.562043,0.203799,39,1.0,False,False,False,...,False,False,False,False,False,True,False,False,False,False
15996,10.174233,289.15,2.0,28.419833,0.939601,39,2.0,True,False,False,...,False,False,False,False,True,True,False,False,False,False
15997,10.959149,101.59,0.0,9.269881,0.266622,22,0.0,False,False,True,...,False,False,True,False,False,False,False,True,False,False
15998,16.209470,219.90,1.0,13.566144,1.025622,27,1.0,False,False,False,...,False,False,False,False,True,False,True,False,False,False


In [32]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [33]:
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.metrics import r2_score

# X_train, X_test, y_train, y_test already banaya hoga

models = {
    "Linear": LinearRegression(),
    "Ridge": Ridge(),
    "Lasso": Lasso(),
    "Decision Tree": DecisionTreeRegressor(),
    "Random Forest": RandomForestRegressor(),
    "Gradient Boosting": GradientBoostingRegressor(),
    "KNN": KNeighborsRegressor(),
    "SVR": SVR()
}

# XGBoost alag import
from xgboost import XGBRegressor
models["XGBoost"] = XGBRegressor()

# Training + Evaluation
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    score = r2_score(y_test, y_pred)
    print(f"{name}: {score:.4f}")

Linear: 0.6092
Ridge: 0.6092
Lasso: 0.6093
Decision Tree: 0.4714
Random Forest: 0.6939
Gradient Boosting: 0.7276
KNN: 0.3618
SVR: 0.0126
XGBoost: 0.7075


In [34]:
from sklearn.ensemble import StackingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.tree import DecisionTreeRegressor
from xgboost import XGBRegressor
from sklearn.metrics import r2_score

# Base models
base_models = [
    ('rf', RandomForestRegressor(n_estimators=100, random_state=42)),
    ('gb', GradientBoostingRegressor()),
    ('dt', DecisionTreeRegressor()),
    ('xgb', XGBRegressor())
]

# Meta model (final learner)
meta_model = LinearRegression()

# Stacking Regressor
stack_model = StackingRegressor(
    estimators=base_models,
    final_estimator=meta_model,
    cv=5
)

# Train
stack_model.fit(X_train, y_train)

# Predict
y_pred = stack_model.predict(X_test)

# Evaluate
score = r2_score(y_test, y_pred)
print("Stacking R2 Score:", score)

Stacking R2 Score: 0.7280247001744462


In [35]:
from sklearn.ensemble import BaggingRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import r2_score

# Base model (weak learner)
base_model = DecisionTreeRegressor()

# Bagging model
bag_model = BaggingRegressor(
    estimator=base_model,
    n_estimators=50,
    max_samples=0.8,
    random_state=42,
    n_jobs=-1
)

# Train
bag_model.fit(X_train, y_train)

# Predict
y_pred = bag_model.predict(X_test)

# Evaluate
score = r2_score(y_test, y_pred)
print("Bagging R2 Score:", score)

Bagging R2 Score: 0.690737525202962


In [36]:
y_pred = stack_model.predict(X_test)

print(y_pred[:10])

[ 3230.32474611  8691.43214107  2746.00172782  3552.01552675
  2859.05769383 12206.16710579  3803.60325841  5893.52802559
  2223.23846341  4334.67185085]


In [37]:
from sklearn.metrics import r2_score, mean_absolute_error,  mean_squared_error, root_mean_squared_error

print("R2 Score:", r2_score(y_test, y_pred))
print("MAE:", mean_absolute_error(y_test, y_pred))
print("MSE:", mean_squared_error(y_test, y_pred))
print("RMSE:", root_mean_squared_error(y_test, y_pred))


R2 Score: 0.7280247001744462
MAE: 878.7074728469254
MSE: 1648914.2527665652
RMSE: 1284.100561781111


In [38]:
import pandas as pd

# Step 1: Create empty dataframe with SAME columns
input_df = pd.DataFrame(columns=X_train.columns)

# Step 2: Add one row (all values = 0)
input_df.loc[0] = 0

# ===== USER INPUT VALUES =====
item_weight = 80
item_mrp = 300
visibility_ratio = 1.4
outlet_age = 30

outlet_size = 'High'
mrp_level = 'Medium'
item_fat = 'Low Fat'
outlet_location = 'Tier 1'
outlet_type = 'Supermarket Type1'
item_category = 'FD'

# ===== Feature Engineering =====
price_per_weight = item_mrp / item_weight
outlet_type_size = outlet_type + "_" + outlet_size

# ===== Fill numeric =====
input_df['Item_Weight'] = item_weight
input_df['Item_MRP'] = item_mrp
input_df['Price_per_Weight'] = price_per_weight
input_df['Visibility_Ratio'] = visibility_ratio
input_df['Outlet_Age'] = outlet_age

# ===== Ordinal Encoding =====
size_map = {'Small': 0, 'Medium': 1, 'High': 2}
mrp_map = {'Low': 0, 'Medium': 1, 'High': 2, 'Very High': 3}

input_df['Outlet_Size'] = size_map[outlet_size]
input_df['MRP_Level'] = mrp_map[mrp_level]

# ===== One-hot encoding =====
def safe_set(col_name):
    if col_name in input_df.columns:
        input_df[col_name] = 1

safe_set(f'Item_Fat_Content_{item_fat}')
safe_set(f'Outlet_Location_Type_{outlet_location}')
safe_set(f'Outlet_Type_{outlet_type}')
safe_set(f'Item_Category_{item_category}')
safe_set(f'Outlet_Type_Size_{outlet_type_size}')

# ===== Prediction =====
prediction = stack_model.predict(input_df)

print("Predicted Sales:", round(prediction[0], 2))

Predicted Sales: 5365.3


In [39]:
import pandas as pd

# ===== HELPER FUNCTIONS =====

def get_menu_choice(prompt, options):
    while True:
        print(f"\n{prompt}")
        for i, opt in enumerate(options, 1):
            print(f"{i}. {opt}")
        
        try:
            choice = int(input("Enter choice number: "))
            if 1 <= choice <= len(options):
                return options[choice - 1]
            else:
                print("⚠️ Invalid choice! Try again.")
        except:
            print("⚠️ Enter a valid number!")

def get_float(prompt, min_val=None, max_val=None):
    while True:
        try:
            val = float(input(prompt))
            if min_val is not None and val <= min_val:
                print(f"⚠️ Value should be > {min_val}")
                continue
            if max_val is not None and val > max_val:
                print(f"⚠️ Value should be <= {max_val}")
                continue
            return val
        except:
            print("⚠️ Invalid input! Please enter a numeric value.")

def get_int(prompt, valid_values=None):
    while True:
        try:
            val = int(input(prompt))
            if valid_values and val not in valid_values:
                print(f"⚠️ Choose from {valid_values}")
                continue
            return val
        except:
            print("⚠️ Invalid input! Enter a number.")

# ===== CREATE EMPTY INPUT =====
input_df = pd.DataFrame(columns=X_train.columns)
input_df.loc[0] = 0

print("\n===== ENTER PRODUCT DETAILS =====")

# ===== NUMERIC INPUT =====
item_weight = get_float("Enter Item Weight (>0): ", 0)
item_mrp = get_float("Enter Item MRP (>0): ", 0)
visibility_ratio = get_float("Enter Visibility Ratio (0–3, Example: 0.8): ", 0, 3)
outlet_age = get_int("Enter Outlet Age (>0): ")

# ===== ORDINAL INPUT =====
print("\nOutlet Size Mapping: Small=0, Medium=1, High=2")
outlet_size = get_int("Enter Outlet Size (0/1/2): ", [0,1,2])

print("\nMRP Level Mapping: Low=0, Medium=1, High=2, Very High=3")
mrp_level = get_int("Enter MRP Level (0/1/2/3): ", [0,1,2,3])

# ===== MENU BASED INPUT =====
item_fat = get_menu_choice(
    "Select Item Fat Content:",
    ["Low Fat", "Regular"]
)

outlet_location = get_menu_choice(
    "Select Outlet Location:",
    ["Tier 1", "Tier 2", "Tier 3"]
)

outlet_type = get_menu_choice(
    "Select Outlet Type:",
    ["Supermarket Type1", "Supermarket Type2", "Supermarket Type3"]
)

item_category = get_menu_choice(
    "Select Item Category:",
    ["DR", "DU", "NC"]
)

# ===== FEATURE ENGINEERING =====
price_per_weight = item_mrp / item_weight
outlet_type_size = outlet_type + "_" + ['Small','Medium','High'][outlet_size]

# ===== FILL NUMERIC =====
input_df['Item_Weight'] = item_weight
input_df['Item_MRP'] = item_mrp
input_df['Price_per_Weight'] = price_per_weight
input_df['Visibility_Ratio'] = visibility_ratio
input_df['Outlet_Age'] = outlet_age

# ===== ORDINAL =====
input_df['Outlet_Size'] = outlet_size
input_df['MRP_Level'] = mrp_level

# ===== ONE-HOT SAFE =====
def safe_set(col_name):
    if col_name in input_df.columns:
        input_df[col_name] = 1

safe_set(f'Item_Fat_Content_{item_fat}')
safe_set(f'Outlet_Location_Type_{outlet_location}')
safe_set(f'Outlet_Type_{outlet_type}')
safe_set(f'Item_Category_{item_category}')
safe_set(f'Outlet_Type_Size_{outlet_type_size}')

# ===== FINAL ORDER =====
input_df = input_df[X_train.columns]

# ===== PREDICTION =====
prediction = stack_model.predict(input_df)

print("\n==========================================")
print(f"🛒 Predicted Annual Sales: ₹{prediction[0]:.2f}")
print("=============================================")


===== ENTER PRODUCT DETAILS =====

Outlet Size Mapping: Small=0, Medium=1, High=2

MRP Level Mapping: Low=0, Medium=1, High=2, Very High=3

Select Item Fat Content:
1. Low Fat
2. Regular

Select Outlet Location:
1. Tier 1
2. Tier 2
3. Tier 3

Select Outlet Type:
1. Supermarket Type1
2. Supermarket Type2
3. Supermarket Type3

Select Item Category:
1. DR
2. DU
3. NC

🛒 Predicted Annual Sales: ₹6270.86
